# Retrieval Agent With Custom Tools

In this notebook, I build a retrieval agent using the Hugging Face `smolagents` library.

The agent answers questions about the guests attending a party by searching through a small knowledge base that I store directly in the notebook. This is a simple form of retrieval: instead of the model guessing an answer, the agent calls a tool that looks up the real information first.

In this notebook, I will learn how to:

- Store a small knowledge base as plain Python data
- Write a keyword based retriever without using any embedding model
- Turn the retriever into an agent tool using the `@tool` decorator
- Rebuild the same tool as a class that inherits from `Tool`
- Combine the retriever with a web search tool so the agent can fall back to the internet
- Let the agent decide which tool to use for each question

I keep everything lightweight on purpose. There is no vector database and no model running on my machine, so the notebook stays fast to run.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` and `Tool` are the two ways of creating a custom tool. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import (
    CodeAgent,
    DuckDuckGoSearchTool,
    InferenceClientModel,
    Tool,
    tool
)

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Building the Guest Knowledge Base

The knowledge base is the information the agent is allowed to look things up in. In a real project this would be a database or a set of documents, but here a list of dictionaries is enough.

Each guest has a name, a relation, a short description and the year they were born. The language model has no way of knowing any of this, so the only way the agent can answer correctly is by calling my retriever tool.

In [ ]:
guests = [
    {
        "name": "Ada Lovelace",
        "relation": "best friend",
        "description": "A respected mathematician and writer, known for her work on Charles Babbage's Analytical Engine. She is often described as the first computer programmer.",
        "born": 1815,
    },
    {
        "name": "Alan Turing",
        "relation": "old university contact",
        "description": "A mathematician and logician who formalised the idea of computation. He enjoys long walks and does not enjoy small talk.",
        "born": 1912,
    },
    {
        "name": "Grace Hopper",
        "relation": "colleague from the navy",
        "description": "A computer scientist and rear admiral who worked on the first compilers. She prefers her coffee black and her meetings short.",
        "born": 1906,
    },
    {
        "name": "Marie Curie",
        "relation": "distant relative",
        "description": "A physicist and chemist who carried out pioneering research on radioactivity. She is the only person to win a Nobel Prize in two different sciences.",
        "born": 1867,
    },
]

print(f"The knowledge base contains {len(guests)} guests.")

## 3. Turning a Guest Record Into Readable Text

A tool has to return text, because the text is what the language model actually reads.

So before writing the retriever I write a small helper that turns one guest dictionary into a clean, readable paragraph. Keeping this in its own function means the retriever stays short and I can change the wording in one place.

In [ ]:
def format_guest(guest: dict) -> str:
    """Turns a single guest record into a readable block of text."""
    return (
        f"Name: {guest['name']}\n"
        f"Relation: {guest['relation']}\n"
        f"Born: {guest['born']}\n"
        f"Description: {guest['description']}"
    )


print(format_guest(guests[0]))

## 4. Writing the Keyword Scoring Function

Now I write the part that actually decides which guest matches a question.

Real retrieval systems compare embeddings, but that means loading a model and storing vectors. For a knowledge base this small, counting shared words works well and costs nothing.

The function lowercases the query and the record, splits both into words, and counts how many words they have in common. A guest whose text shares more words with the question gets a higher score.

In [ ]:
import re


def tokenize(text: str) -> set:
    """Splits text into a set of lowercase words."""
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def score_guest(query: str, guest: dict) -> int:
    """Counts how many words the query and the guest record share."""
    query_words = tokenize(query)
    guest_words = tokenize(format_guest(guest))
    return len(query_words & guest_words)


for guest in guests:
    print(guest["name"], "->", score_guest("Tell me about the mathematician Ada", guest))

## 5. Ranking the Guests

With a score for each guest, retrieval is just sorting.

I sort every guest by score, drop the ones that share no words at all with the question, and keep the best few. Returning more than one result is useful because the agent can read them and pick the one that really answers the question.

In [ ]:
def search_guests(query: str, top_k: int = 2) -> list:
    """Returns the guests whose records best match the query."""
    scored = [(score_guest(query, guest), guest) for guest in guests]
    matches = [(score, guest) for score, guest in scored if score > 0]
    matches.sort(key=lambda pair: pair[0], reverse=True)
    return [guest for score, guest in matches[:top_k]]


for guest in search_guests("Who worked on compilers?"):
    print(guest["name"])

## 6. Improving the Scores by Ignoring Common Words

Testing the scoring function showed a problem. Words like "the", "and" or "a" appear in almost every record, so a guest can pick up points from a question that has nothing to do with them.

The fix is a stopword list. I remove these very common words before comparing, so only the meaningful words count. After this change the scores are lower but much more honest.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "the", "is", "was", "are", "were", "of", "on", "in",
    "to", "for", "with", "who", "what", "which", "me", "my", "about",
    "tell", "her", "his", "she", "he", "it", "at", "as", "by", "that",
}


def tokenize(text: str) -> set:
    """Splits text into a set of lowercase words, ignoring common words."""
    words = set(re.findall(r"[a-z0-9]+", text.lower()))
    return words - STOPWORDS


for guest in guests:
    print(guest["name"], "->", score_guest("Tell me about the mathematician Ada", guest))

## 7. Turning the Retriever Into a Tool

The retriever works, but the agent cannot use a plain Python function. It needs a tool.

The `@tool` decorator does the conversion for me. The important detail is the docstring: `smolagents` reads it to build the tool description that the model sees, so the description and the argument list have to explain clearly when this tool should be used.

In [ ]:
@tool
def guest_info_tool(query: str) -> str:
    """
    Looks up information about the guests invited to the party.

    Args:
        query (str): A name or a few words describing the guest to look up.
    """
    matches = search_guests(query)

    if not matches:
        return "No guest matching that description is on the invitation list."

    return "\n\n".join(format_guest(guest) for guest in matches)

## 8. Testing the Tool on Its Own

Before giving the tool to an agent I call it directly.

This is worth doing every time. If the tool returns something wrong here, the agent will only make the problem harder to see, because then I cannot tell whether the mistake came from my code or from the model.

In [ ]:
print(guest_info_tool("Who is Ada Lovelace?"))
print()
print(guest_info_tool("Tell me about the guest who studied radioactivity"))
print()
print(guest_info_tool("Is Napoleon coming?"))

## 9. Creating the Retrieval Agent

Now I give the tool to a `CodeAgent`.

The agent receives the question, decides that it needs guest information, writes a line of Python that calls `guest_info_tool`, reads the result and then writes the final answer in its own words.

In [ ]:
guest_agent = CodeAgent(
    tools=[guest_info_tool],
    model=model
)

## 10. Asking the Agent a Question

I ask a question that the model cannot possibly answer from memory, because the relation between me and the guest only exists in my own knowledge base.

If the answer mentions that Ada Lovelace is my best friend, then the agent really did call the tool instead of relying on what the model already knew.

In [ ]:
guest_agent.run(
    "Tell me about Ada Lovelace and explain how she is related to me."
)

## 11. Rebuilding the Same Tool as a Class

The `@tool` decorator is the quickest way to make a tool, but it has a limit: the tool can only work with data that already exists outside it, like my global `guests` list.

Inheriting from `Tool` solves that. The class holds its own copy of the guest list, so the same tool can be reused with a completely different set of guests. The class also states its `name`, `description`, `inputs` and `output_type` explicitly instead of taking them from a docstring.

In [ ]:
class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = (
        "Retrieves information about the guests invited to the party, "
        "including their relation to the host and a short description."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "A name or a few words describing the guest to look up.",
        }
    }
    output_type = "string"

    def __init__(self, guest_list: list):
        super().__init__()
        self.guest_list = guest_list

    def forward(self, query: str) -> str:
        scored = [(score_guest(query, guest), guest) for guest in self.guest_list]
        matches = sorted(
            [pair for pair in scored if pair[0] > 0],
            key=lambda pair: pair[0],
            reverse=True,
        )

        if not matches:
            return "No guest matching that description is on the invitation list."

        return "\n\n".join(format_guest(guest) for score, guest in matches[:2])

## 12. Reusing the Tool With a Different Guest List

This is the advantage of the class version. I create a second, completely separate guest list and build a tool from it without touching any of the code above.

The two tools behave the same way but search different data, which would not be possible with the decorator version.

In [ ]:
other_guests = [
    {
        "name": "Katherine Johnson",
        "relation": "guest of honour",
        "description": "A mathematician whose orbital calculations were critical to the first American crewed spaceflights.",
        "born": 1918,
    },
    {
        "name": "Tim Berners-Lee",
        "relation": "neighbour",
        "description": "The computer scientist who invented the World Wide Web while working at CERN.",
        "born": 1955,
    },
]

party_tool = GuestInfoRetrieverTool(guests)
other_tool = GuestInfoRetrieverTool(other_guests)

print(other_tool("Who invented the web?"))

## 13. Adding a Web Search Tool

My knowledge base only knows who is invited and how they are related to me. It says nothing about the wider world.

So I add `DuckDuckGoSearchTool` as a second source of information. The interesting part is that I do not tell the agent which tool to use. The agent reads both tool descriptions and decides for itself: the guest list for anything about the invitation, the web for anything else.

In [ ]:
search_tool = DuckDuckGoSearchTool()

print(search_tool("Who was Katherine Johnson?"))

## 14. Adding a Conversation Starter Tool

To make the agent genuinely useful at the party, I add a third tool that suggests something to talk about based on a topic.

This tool is deliberately simple. Its value in this notebook is that it gives the agent a third option to choose between, which makes the tool selection step easier to observe.

In [ ]:
@tool
def conversation_starter(topic: str) -> str:
    """
    Suggests a question to start a conversation about a given topic.

    Args:
        topic (str): The subject the guest is known for, such as mathematics or physics.
    """
    starters = {
        "mathematics": "Ask which proof they find the most beautiful and why.",
        "computing": "Ask what they think the next big shift in computing will be.",
        "physics": "Ask which experiment surprised them the most.",
        "chemistry": "Ask how they first became interested in the subject.",
    }

    return starters.get(
        topic.lower(),
        "Ask them what they are working on at the moment and listen carefully.",
    )

## 15. Building the Complete Agent

Now I put all three tools together into one agent.

I use the class based retriever here rather than the decorator version, because it carries its own guest list and is the one I would reuse in a real project.

In [ ]:
party_agent = CodeAgent(
    tools=[party_tool, search_tool, conversation_starter],
    model=model
)

## 16. Giving the Agent a Task That Needs Several Tools

This question cannot be answered with one tool alone.

The agent has to look up Grace Hopper in my guest list to find out what she is known for, and then use the conversation starter tool for that subject. Watching the steps it prints is the clearest way to see an agent actually planning rather than just answering.

In [ ]:
party_agent.run(
    "One of my guests worked on the first compilers. "
    "Who is she, and what should I talk to her about?"
)